# Planejamento de rotas com *Osmnx* + *NetworkX*

### Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
import folium
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from folium.plugins import MeasureControl, Fullscreen
import warnings
warnings.filterwarnings('ignore')


### Carregamos e inspecionamos os dados

**Carregamos o dataset**

In [ ]:
gdf = pd.read_csv("lista_bares.csv")
X = np.array(gdf[['latitude', 'longitude']])

**Inspecionamos os dados**

In [ ]:
print("=== Informações do dataset ===\n")
gdf.info()

print("\n=== Primeiras 5 coordenadas (lat, lon) ===\n")
print(X[:5])

print(f"\nTotal de bares carregados: {len(gdf)}")
print(f"Colunas disponíveis: {list(gdf.columns)}")

### Definimos o roteiro mais curto

In [ ]:
# Calcular a distância para cada par de pontos consecutivos

bares = gdf['Name']
nbrs = NearestNeighbors(n_neighbors=len(X), algorithm='ball_tree').fit(X)
distances, indices = nbrs.kneighbors(X)

# Encontrar o roteiro mais curto
visited = np.zeros(len(X), dtype=bool)

end_point = 5  # Definindo o ponto final como 13 (Casa d'Itália)

visited[0] = True
tour = [0]
current = 0

# Modificado para parar quando chegar ao ponto 39
while current != end_point and len(tour) < len(X):
    unvisited_mask = np.logical_not(visited[indices[current]])
    if np.any(unvisited_mask):
        nearest = indices[current][unvisited_mask][0].item()
    else:
        # Se todos os vizinhos foram visitados, escolha o próximo não visitado
        unvisited = np.where(visited == False)[0]
        if len(unvisited) > 0:
            nearest = unvisited[0]
        else:
            break
    
    tour.append(nearest)
    visited[nearest] = True
    current = nearest

    # Se chegou ao ponto final, pare
    if current == end_point:
        break

# Resultado
print("Rota mais curta terminando no item 5:")
for i, point in enumerate(tour):
    print(f"{i}. {bares[point]} (Ponto {point})")


**Criamos o dicionário das coordenadas**

In [ ]:
# Versão concisa usando dict comprehension
coordenadas_referencia = {row['Name']: (row['latitude'], row['longitude']) 
                          for idx, row in gdf.iterrows()}

# Nomes dos locais na ordem original
bares = gdf['Name'].tolist()  

# Ordenar o dicionário conforme a rota
coordenadas_ordenadas = {
    bares[i]: coordenadas_referencia[bares[i]] 
    for i in tour
}

print(coordenadas_ordenadas)

### Definimos os parâmetros

In [ ]:
# Modo de transporte: 'walk' (pedestre) ou 'drive' (carro)
MODO_TRANSPORTE = 'drive'  # Altere para 'drive' se quiser rotas de carro

# Distância máxima para baixar o grafo (em metros)
# Aumente se os bares estiverem mais distantes
DISTANCIA_GRAFO = 30000  # 2km ao redor do centro

# Número de roteiros (clusters)
NUMERO_ROTEIROS = 7

print(f"\n⚙️ Configurações:")
print(f"   Modo de transporte: {MODO_TRANSPORTE.upper()}")
print(f"   Raio do grafo: {DISTANCIA_GRAFO} m")
print(f"   Número de roteiros: {NUMERO_ROTEIROS}")


### Definimos a função para *clustering*

In [ ]:
def clusterizar_bares(coordenadas, n_clusters=4):
    """
    Agrupa os bares em clusters baseado na proximidade geográfica
    """
    nomes = list(coordenadas.keys())
    coords = np.array(list(coordenadas.values()))
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(coords)
    
    clusters = {}
    for i, nome in enumerate(nomes):
        cluster_id = labels[i]
        if cluster_id not in clusters:
            clusters[cluster_id] = {}
        clusters[cluster_id][nome] = tuple(coords[i])
    
    return clusters


### Definimos a função para extrair o grafo das ruas

In [ ]:
def obter_grafo_cluster(bares_cluster, modo='walk'):
    """
    Obtém o grafo de ruas para um cluster de bares
    """
    # Calcular centro do cluster
    coords = list(bares_cluster.values())
    center_lat = np.mean([c[0] for c in coords])
    center_lon = np.mean([c[1] for c in coords])
    
    # Definir network_type baseado no modo
    network_type = 'walk' if modo == 'walk' else 'drive'
    
    try:
        # Baixar grafo ao redor do centro
        G = ox.graph_from_point(
            (center_lat, center_lon), 
            dist=DISTANCIA_GRAFO, 
            network_type=network_type,
            simplify=True
        )
        print(f"      Grafo baixado: {len(G.nodes)} nós, {len(G.edges)} arestas")
        return G
    except Exception as e:
        print(f"      ⚠️ Erro ao baixar grafo: {e}")
        return None


### Definimos funções para cálculo de distâncias

In [ ]:
def calcular_distancias_reais(bares_cluster, G):
    """
    Calcula matriz de distâncias reais usando o grafo OSMnx
    """
    if G is None:
        # Fallback para Haversine
        print("Usando Haversine (fallback)")
        return calcular_distancias_haversine(bares_cluster)
    
    nomes = list(bares_cluster.keys())
    coords = list(bares_cluster.values())
    n = len(coords)
    
    # Encontrar nós mais próximos para cada bar
    nodes = []
    for lat, lon in coords:
        try:
            node = ox.distance.nearest_nodes(G, lon, lat)
            nodes.append(node)
        except:
            nodes.append(None)
    
    # Calcular matriz de distâncias
    dist_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
            if nodes[i] is not None and nodes[j] is not None:
                try:
                    # Distância real no grafo
                    dist = nx.shortest_path_length(G, nodes[i], nodes[j], weight='length')
                    dist_matrix[i, j] = dist
                    dist_matrix[j, i] = dist
                except:
                    # Fallback para Haversine
                    lat1, lon1 = coords[i]
                    lat2, lon2 = coords[j]
                    dist = haversine_distance(lat1, lon1, lat2, lon2)
                    dist_matrix[i, j] = dist
                    dist_matrix[j, i] = dist
            else:
                lat1, lon1 = coords[i]
                lat2, lon2 = coords[j]
                dist = haversine_distance(lat1, lon1, lat2, lon2)
                dist_matrix[i, j] = dist
                dist_matrix[j, i] = dist
    
    return dist_matrix

def calcular_distancias_haversine(bares_cluster):
    """
    Calcula matriz de distâncias Haversine (fallback)
    """
    coords = list(bares_cluster.values())
    n = len(coords)
    dist_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i+1, n):
            lat1, lon1 = coords[i]
            lat2, lon2 = coords[j]
            dist = haversine_distance(lat1, lon1, lat2, lon2)
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
    
    return dist_matrix


### Definimos a função para extrair a geometria da rota

In [ ]:
def obter_geometria_rota(bares_cluster, rota_indices, G):
    """
    Obtém a geometria completa da rota para visualização
    """
    if G is None:
        return []
    
    nomes = list(bares_cluster.keys())
    coords = list(bares_cluster.values())
    
    # Encontrar nós
    nodes = []
    for lat, lon in coords:
        try:
            node = ox.distance.nearest_nodes(G, lon, lat)
            nodes.append(node)
        except:
            nodes.append(None)
    
    geometrias = []
    for k in range(len(rota_indices)-1):
        i = rota_indices[k]
        j = rota_indices[k+1]
        
        if nodes[i] is not None and nodes[j] is not None:
            try:
                # Obter caminho mais curto
                path = nx.shortest_path(G, nodes[i], nodes[j], weight='length')
                # Converter para coordenadas
                coords_path = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]
                geometrias.append(coords_path)
            except:
                # Fallback: linha reta
                lat1, lon1 = coords[i]
                lat2, lon2 = coords[j]
                geometrias.append([[lat1, lon1], [lat2, lon2]])
        else:
            lat1, lon1 = coords[i]
            lat2, lon2 = coords[j]
            geometrias.append([[lat1, lon1], [lat2, lon2]])
    
    return geometrias


### Definimos a função para otimizar as rotas

In [ ]:
def otimizar_rota_cluster(bares_cluster, G):
    """
    Otimiza rota dentro de um cluster usando TSP
    """
    n = len(bares_cluster)
    
    if n <= 1:
        nomes = list(bares_cluster.keys())
        return nomes, 0.0, []
    
    # Calcular matriz de distâncias reais
    dist_matrix = calcular_distancias_reais(bares_cluster, G)
    
    # TSP guloso
    visitados = [0]
    atual = 0
    distancia_total = 0.0
    
    while len(visitados) < n:
        melhor_dist = float('inf')
        melhor_idx = -1
        
        for j in range(n):
            if j not in visitados:
                if dist_matrix[atual, j] < melhor_dist:
                    melhor_dist = dist_matrix[atual, j]
                    melhor_idx = j
        
        if melhor_idx != -1:
            distancia_total += melhor_dist
            visitados.append(melhor_idx)
            atual = melhor_idx
        else:
            break
    
    # Obter geometria da rota
    geometrias = obter_geometria_rota(bares_cluster, visitados, G)
    
    # Converter para nomes
    nomes = list(bares_cluster.keys())
    rota_ordenada = [nomes[i] for i in visitados]
    
    return rota_ordenada, distancia_total, geometrias


### Definimos a função para criar o mapa

In [ ]:
def criar_mapa_clusters(todos_clusters, rotas, distancias, geometrias, modo):
    """
    Cria mapa com múltiplos clusters e rotas reais
    """
    cores = {
        0: '#e41a1c',  # vermelho
        1: '#377eb8',  # azul
        2: '#4daf4a',  # verde
        3: '#984ea3',  # roxo
        4: '#ff7f00',  # laranja
        5: '#ffff33',  # amarelo
        6: '#a65628',  # marrom
        7: '#f781bf',  # rosa
    }
    
    # Calcular centro do mapa
    todas_coords = []
    for cluster in todos_clusters.values():
        for coord in cluster.values():
            todas_coords.append(coord)
    todas_coords = np.array(todas_coords)
    center_lat = todas_coords[:, 0].mean()
    center_lon = todas_coords[:, 1].mean()
    
    # Criar mapa base
    mapa = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB positron')
    folium.TileLayer('CartoDB dark_matter', name='Mapa Escuro', show=False).add_to(mapa)
    
    for cluster_id, bares in todos_clusters.items():
        cor = cores.get(cluster_id, '#999999')
        rota_nomes = rotas.get(cluster_id, [])
        rota_geometrias = geometrias.get(cluster_id, [])
        distancia = distancias.get(cluster_id, 0)
        
        fg = folium.FeatureGroup(name=f'Roteiro {cluster_id + 1} - {len(bares)} bares')
        
        # Adicionar rotas (seguindo ruas reais)
        for geom in rota_geometrias:
            if geom and len(geom) > 1:
                folium.PolyLine(
                    geom,
                    color=cor,
                    weight=4,
                    opacity=0.8,
                    popup=f'Trecho do Roteiro {cluster_id + 1}'
                ).add_to(fg)
        
        # Adicionar marcadores
        for idx, nome in enumerate(rota_nomes):
            coord = bares[nome]
            
            if idx == 0:
                cor_marcador = 'green'
                icone = 'play'
                tooltip = f"🏁 INÍCIO: {nome}"
            elif idx == len(rota_nomes) - 1:
                cor_marcador = 'red'
                icone = 'stop'
                tooltip = f"🏁 FIM: {nome}"
            else:
                cor_marcador = cores.get(cluster_id, 'blue')
                icone = 'beer'
                tooltip = nome
            
            popup_text = f"""
            <div style="font-family: Arial; width: 220px;">
                <b>{nome}</b><br>
                📍 Parada {idx + 1} de {len(rota_nomes)}<br>
                🍺 Roteiro {cluster_id + 1}<br>
            </div>
            """
            
            if idx == 0:
                popup_text = f"""
                <div style="font-family: Arial; width: 220px;">
                    <b>{nome}</b><br>
                    📍 Parada {idx + 1} de {len(rota_nomes)}<br>
                    🍺 Roteiro {cluster_id + 1}<br>
                    📏 Distância total: {distancia/1000:.2f} km<br>
                    🚶 Modo: {modo.upper()}
                </div>
                """
            
            folium.Marker(
                coord,
                popup=folium.Popup(popup_text, max_width=300),
                icon=folium.Icon(color=cor_marcador, icon=icone, prefix='fa'),
                tooltip=tooltip
            ).add_to(fg)
        
        fg.add_to(mapa)
    
    # Adicionar controles
    folium.LayerControl().add_to(mapa)
    MeasureControl().add_to(mapa)
    Fullscreen().add_to(mapa)
    
    # Estatísticas totais
    total_bares = sum(len(c) for c in todos_clusters.values())
    total_distancia = sum(distancias.values())
    tempo_estimado = total_distancia / (1.4 if modo == 'walk' else 8.3) / 60
    
    # Legenda
    legenda_html = f'''
    <div style="position: fixed; bottom: 10px; right: 10px; z-index: 1000;
                background-color: white; padding: 12px; border-radius: 8px;
                border: 2px solid #ff6b6b; font-family: 'Segoe UI', Arial, sans-serif;
                font-size: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.2);
                max-width: 250px;">
        <h4 style="margin:0 0 8px 0; color:#ff6b6b;">🍺 ROTEIROS POR PROXIMIDADE</h4>
        <b>🚶 Modo:</b> {'PEDESTRE' if modo == 'walk' else 'CARRO'}<br>
        <b>📍 Total de bares:</b> {total_bares}<br>
        <b>🗺️ Número de roteiros:</b> {len(todos_clusters)}<br>
        <b>📏 Distância total:</b> {total_distancia/1000:.2f} km<br>
        <b>⏱️ Tempo estimado:</b> {tempo_estimado:.0f} min<br>
        <hr style="margin: 5px 0;">
        <small>
        🟢 Verde: Início do roteiro<br>
        🔴 Vermelho: Fim do roteiro<br>
        🔵 Demais cores: diferentes roteiros<br>
        📍 Clique nos marcadores para detalhes
        </small>
    </div>
    '''
    mapa.get_root().html.add_child(folium.Element(legenda_html))
    
    return mapa


### Execução final

In [ ]:
print(f"\n📊 Dados de entrada:")
print(f"   Total de locais: {len(coordenadas_ordenadas)}")

# 1. Clusterizar bares
print(f"\n📊 Agrupando bares em {NUMERO_ROTEIROS} roteiros...")
clusters = clusterizar_bares(coordenadas_ordenadas, n_clusters=NUMERO_ROTEIROS)

# 2. Para cada cluster, otimizar rota
print(f"\n🚀 Otimizando rotas com OSMnx (modo: {MODO_TRANSPORTE})...")
todas_rotas = {}
todas_distancias = {}
todas_geometrias = {}
todos_grafos = {}

for cluster_id, bares in clusters.items():
    print(f"\n   Roteiro {cluster_id + 1}: {len(bares)} bares")
    
    # Obter grafo de ruas para este cluster
    G = obter_grafo_cluster(bares, modo=MODO_TRANSPORTE)
    todos_grafos[cluster_id] = G
    
    # Otimizar rota
    rota, distancia, geometrias = otimizar_rota_cluster(bares, G)
    
    todas_rotas[cluster_id] = rota
    todas_distancias[cluster_id] = distancia
    todas_geometrias[cluster_id] = geometrias
    
    print(f"      Distância total: {distancia/1000:.2f} km")
    if len(rota) <= 5:
        print(f"      Rota: {' → '.join(rota)}")
    else:
        print(f"      Rota: {' → '.join(rota[:3])} ... → {rota[-1]}")

# 3. Criar mapa
print(f"\n🗺️ Criando mapa interativo...")
mapa_final = criar_mapa_clusters(
    clusters, todas_rotas, todas_distancias, 
    todas_geometrias, MODO_TRANSPORTE
)

# 4. Salvar mapa
mapa_final.save('roteiros_osmnx_clusters.html')
print(f"\n✅ Mapa salvo como 'roteiros_osmnx_clusters.html'")

# 5. Relatório final
print("\n" + "="*60)
print("📋 RELATÓRIO FINAL")
print("="*60)

total_bares = sum(len(c) for c in clusters.values())
total_distancia = sum(todas_distancias.values())

print(f"\n📊 Resumo por roteiro:")
for cluster_id in sorted(clusters.keys()):
    print(f"   Roteiro {cluster_id + 1}: {len(clusters[cluster_id])} bares, "
          f"{todas_distancias[cluster_id]/1000:.2f} km")

print(f"\n📈 TOTAL:")
print(f"   Total de bares: {total_bares}")
print(f"   Total de roteiros: {len(clusters)}")
print(f"   Distância total: {total_distancia/1000:.2f} km")

print("\n🎉 ANÁLISE CONCLUÍDA!")

# Exibir mapa (se estiver no Jupyter)
#mapa_final

### Visualizamos o mapa interativo

In [ ]:
# Exibir mapa
mapa_final